# Oil Prices in 2026 — Act 5: What If the Model Could Read the News?

The [companion notebook](energy_oil_case_study.ipynb) showed that Prophet's rolling 30-day
forecast catastrophically missed the 2026 oil price surge — forecasting ~$61/bbl
while WTI hit $100. The model wasn't wrong in principle; it simply had no mechanism
for incorporating the geopolitical context that was publicly available at the time.

This notebook asks: **could a context-aware LLM forecaster have done better?**

We evaluate three key forecast origins in early 2026 using three methods side by side:
- **Prophet** (baseline — already computed, loaded from cache)
- **LLMP — no context** (Gemini 3 Flash, history only)
- **LLMP — with context** (same model + plausibly-knowable geopolitical context at each origin)

And we frame the comparison three ways:

| | Question type | Evaluation |
|---|---|---|
| **Act 5** | *Trajectory* — what will the 30-day price path look like? | MAE vs. actuals |
| **Act 6** | *Binary* — will price exceed a meaningful threshold in 30 days? | Calibration of P(exceed) |
| **Act 7** | *Causal* — what forces are the model anchoring on? | Qualitative reasoning audit |

In [15]:
from __future__ import annotations

import json
import logging
import os
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.subplots as psp
from dotenv import load_dotenv

warnings.filterwarnings("ignore")
logging.getLogger("prophet").setLevel(logging.ERROR)

# ── Repo root: walk up from CWD until pyproject.toml is found ─────────────────
_cwd = Path(os.getcwd()).resolve()
REPO_ROOT = _cwd
while not (REPO_ROOT / "pyproject.toml").exists():
    if REPO_ROOT.parent == REPO_ROOT:
        REPO_ROOT = _cwd
        break
    REPO_ROOT = REPO_ROOT.parent

DATA_DIR = REPO_ROOT / "data"

for p in [str(REPO_ROOT / "implementations"), str(REPO_ROOT / "aieng-forecasting")]:
    if p not in sys.path:
        sys.path.insert(0, p)

load_dotenv(REPO_ROOT / ".env")

# ── Colour palette (matches companion notebook) ────────────────────────────────
CLR_HISTORY   = "#bdd7e7"
CLR_ACTUAL    = "#2171b5"   # solid blue — the truth
CLR_PROPHET   = "#636363"   # grey — the blind baseline
CLR_LLMP_BARE = "#fd8d3c"   # orange — LLM, history only
CLR_LLMP_CTX  = "#2ca02c"   # green — LLM, with context
CLR_CONFLICT  = "#d62728"   # red — conflict annotation
CONFLICT_DATE = pd.Timestamp("2026-03-01")

print(f"Repo root : {REPO_ROOT}")
print(f"Data dir  : {DATA_DIR}")
print("Setup complete.")

Repo root : /Users/ethanjackson/agentic-forecasting
Data dir  : /Users/ethanjackson/agentic-forecasting/data
Setup complete.


In [16]:
# ── WTI price history (same cache as companion notebook) ──────────────────────
PRICE_CACHE        = DATA_DIR / "wti_price_history.parquet"
PROPHET_CACHE      = DATA_DIR / "energy_case_study_forecasts_30d_daily_v3.parquet"
PROPHET_TRAJ_CACHE = DATA_DIR / "energy_prophet_trajectories.parquet"
LLMP_CACHE         = DATA_DIR / "energy_llmp_context_forecasts.parquet"

price_df = pd.read_parquet(PRICE_CACHE)
price_df.index = pd.DatetimeIndex(
    [pd.Timestamp(str(d)[:10]) for d in price_df.index]
)
price_df.index.name = "date"
price_df = price_df.sort_index()

prophet_df = pd.read_parquet(PROPHET_CACHE)
prophet_df["sim_day"]        = pd.to_datetime(prophet_df["sim_day"])
prophet_df["resolution_date"] = pd.to_datetime(prophet_df["resolution_date"])

print(f"WTI price history : {price_df.index[0].date()} → {price_df.index[-1].date()} ({len(price_df):,} days)")
print(f"Prophet forecasts : {prophet_df['sim_day'].min().date()} → {prophet_df['sim_day'].max().date()} ({len(prophet_df):,} rows)")

WTI price history : 2021-01-04 → 2026-05-01 (1,340 days)
Prophet forecasts : 2025-01-02 → 2026-04-01 (314 rows)


---

## The Setup

We pick **three forecast origins** in early 2026 — each representing a different
stage of the geopolitical escalation that drove WTI from ~$58 to $100+ between
January and April 2026.

| Origin | WTI at origin | Resolution date | Actual WTI at resolution | Prophet forecast | Context available |
|---|---|---|---|---|---|
| Jan 5, 2026 | $58 | Feb 4, 2026 | **$65** | $58 (inside CI) | Tensions building; OPEC+ cuts; insurance premiums rising |
| Feb 2, 2026 | $62 | Mar 4, 2026 | **$75** | $61 (miss — above CI) | Gulf of Oman incident; escalation fears; analyst upgrades |
| Mar 2, 2026 | $71 | Apr 1, 2026 | **$100** | $61 (catastrophic miss) | Conflict active; Strait of Hormuz blockade; IEA emergency session |

For each origin we ask: does an LLMP with access to publicly-available context
shift its forecast in the right direction — toward the actual outcome?

In [17]:
def compress_history(
    price_df: pd.DataFrame,
    as_of: pd.Timestamp,
    recent_window_months: int = 6,
) -> pd.DataFrame:
    """Return a token-efficient history: weekly averages for older data, daily for recent.

    Compresses ~1300 daily rows to ~200 rows while preserving the recent
    daily granularity that matters most for the LLM's short-horizon forecast.
    """
    hist = price_df[price_df.index <= as_of].copy()
    cutoff_daily = as_of - pd.DateOffset(months=recent_window_months)

    older = (
        hist[hist.index < cutoff_daily]
        .resample("W")
        .mean()
        .reset_index()
        .rename(columns={"date": "timestamp", "price": "value"})
    )
    recent = (
        hist[hist.index >= cutoff_daily]
        .reset_index()
        .rename(columns={"date": "timestamp", "price": "value"})
    )

    result = pd.concat([older, recent], ignore_index=True)
    result["timestamp"] = pd.to_datetime(result["timestamp"])
    return result[["timestamp", "value"]].sort_values("timestamp").reset_index(drop=True)


def prophet_row_at_origin(prophet_df: pd.DataFrame, origin: pd.Timestamp) -> pd.Series:
    """Return the Prophet forecast row whose sim_day is nearest to (on or after) origin."""
    candidates = prophet_df[prophet_df["sim_day"] >= origin]
    return candidates.iloc[0]


def resolution_price(price_df: pd.DataFrame, origin: pd.Timestamp, horizon_calendar_days: int = 30) -> tuple[pd.Timestamp, float]:
    """Return (resolution_date, actual_price) for a calendar-day horizon from origin."""
    target = origin + pd.Timedelta(days=horizon_calendar_days)
    row = price_df[price_df.index >= target].iloc[0]
    return row.name, float(row["price"])


# Sanity-check the three origins
ORIGINS = [
    pd.Timestamp("2026-01-05"),
    pd.Timestamp("2026-02-02"),
    pd.Timestamp("2026-03-02"),
]

print("Origin summary:")
for o in ORIGINS:
    price_at_origin = float(price_df[price_df.index >= o].iloc[0]["price"])
    res_date, res_price = resolution_price(price_df, o)
    p_row = prophet_row_at_origin(prophet_df, o)
    print(
        f"  {o.date()}  WTI=${price_at_origin:.2f}  "
        f"→ resolution {res_date.date()} actual=${res_price:.2f}  "
        f"prophet=${p_row['yhat']:.2f} [{p_row['yhat_lower']:.1f},{p_row['yhat_upper']:.1f}]  "
        f"inside_ci={p_row['inside_ci']}"
    )

Origin summary:
  2026-01-05  WTI=$58.32  → resolution 2026-02-04 actual=$65.14  prophet=$57.55 [49.5,65.5]  inside_ci=True
  2026-02-02  WTI=$62.14  → resolution 2026-03-04 actual=$74.66  prophet=$60.91 [52.8,69.4]  inside_ci=False
  2026-03-02  WTI=$71.23  → resolution 2026-04-01 actual=$100.12  prophet=$61.32 [53.3,69.3]  inside_ci=False


In [18]:
# ── Prophet full-trajectory forecasts for the three origins ──────────────────
#
# The existing Prophet cache stores only the terminal 30-calendar-day-ahead
# point per origin.  Here we re-run Prophet for just the three key origins and
# produce a 21-business-day trajectory fan — matching the LLMP output structure
# so the trajectory chart shows a like-for-like comparison between methods.
#
# Fitting 3 Prophet models takes ~10 s; results are cached to
# data/energy_prophet_trajectories.parquet so subsequent runs are instant.

from prophet import Prophet  # type: ignore[import-untyped]


def _fit_prophet_at_origin(price_df: pd.DataFrame, origin: pd.Timestamp) -> pd.DataFrame:
    """Fit one Prophet model on all data up to origin; return 21-business-day trajectory."""
    train_df = price_df.loc[:origin][["price"]].reset_index()
    train_df.columns = pd.Index(["ds", "y"])

    model = Prophet(
        interval_width=0.95,
        daily_seasonality=False,
        weekly_seasonality=False,
        yearly_seasonality=True,
        seasonality_mode="multiplicative",
    )
    model.fit(train_df)

    # Predict enough calendar days to cover 21 business days (≈ 31 calendar days)
    future = model.make_future_dataframe(periods=35, freq="D")
    pred   = model.predict(future).set_index("ds")

    bday_dates = pd.bdate_range(start=origin + pd.offsets.BDay(1), periods=21)
    rows = []
    for h, date in enumerate(bday_dates, start=1):
        cal_date = date.normalize()
        if cal_date in pred.index:
            row = pred.loc[cal_date]
        else:
            nearest_idx = int((pred.index - cal_date).abs().argmin())
            row = pred.iloc[nearest_idx]
        rows.append({
            "origin":        origin,
            "forecast_date": date,
            "horizon":       h,
            "yhat":          float(row["yhat"]),
            "yhat_lower":    float(row["yhat_lower"]),
            "yhat_upper":    float(row["yhat_upper"]),
        })

    return pd.DataFrame(rows)


def load_prophet_trajectories(
    price_df: pd.DataFrame,
    origins: list[pd.Timestamp],
    cache_path: Path,
) -> pd.DataFrame:
    """Load from cache or compute full Prophet trajectory for each origin."""
    if cache_path.exists():
        df = pd.read_parquet(cache_path)
        df["origin"]        = pd.to_datetime(df["origin"])
        df["forecast_date"] = pd.to_datetime(df["forecast_date"])
        print(f"Loaded {len(df)} Prophet trajectory rows from cache.")
        return df

    print("Fitting Prophet at 3 origins (~10 s)...")
    frames = []
    for origin in origins:
        print(f"  {origin.date()} ...", end=" ", flush=True)
        frames.append(_fit_prophet_at_origin(price_df, origin))
        print("done")

    df = pd.concat(frames, ignore_index=True)
    df.to_parquet(cache_path, index=False)
    print(f"Saved {len(df)} rows to {cache_path}")
    return df


prophet_traj_df = load_prophet_trajectories(price_df, ORIGINS, PROPHET_TRAJ_CACHE)
prophet_traj_df.head(6)

Fitting Prophet at 3 origins (~10 s)...
  2026-01-05 ... 

06:17:38 - cmdstanpy - INFO - Chain [1] start processing
06:17:39 - cmdstanpy - INFO - Chain [1] done processing


done
  2026-02-02 ... 

06:17:39 - cmdstanpy - INFO - Chain [1] start processing
06:17:39 - cmdstanpy - INFO - Chain [1] done processing


done
  2026-03-02 ... 

06:17:39 - cmdstanpy - INFO - Chain [1] start processing
06:17:39 - cmdstanpy - INFO - Chain [1] done processing


done
Saved 63 rows to /Users/ethanjackson/agentic-forecasting/data/energy_prophet_trajectories.parquet


,origin,forecast_date,horizon,yhat,yhat_lower,yhat_upper
0,2026-01-05,2026-01-06,1,56.110597,47.564043,63.171764
1,2026-01-05,2026-01-07,2,56.226107,48.657983,64.782960
2,2026-01-05,2026-01-08,3,56.346479,48.503792,64.105882
3,2026-01-05,2026-01-09,4,56.470931,47.860042,64.401306
4,2026-01-05,2026-01-12,5,56.857824,48.616767,64.700857
5,2026-01-05,2026-01-13,6,56.986786,48.967157,65.217871


In [19]:
# ── Context snippets — plausibly knowable on each origin date ─────────────────
# These represent the kind of information a professional energy analyst
# would have had access to from public sources: news, vessel-tracking
# services, futures data, and analyst reports published before the origin date.

ORIGIN_CONTEXTS: dict[str, dict] = {
    "2026-01-05": {
        "label": "Jan 5, 2026",
        "threshold_usd": 65.0,
        "context_text": (
            "As of January 5 2026:\n"
            "- WTI crude has been range-bound in the $56–62 band since October 2025 on soft "
            "demand signals and elevated US inventory builds.\n"
            "- OPEC+ is maintaining its current production-cut agreement through Q1 2026; "
            "no rollback has been signalled.\n"
            "- Iranian proxy forces conducted three separate attacks on US logistics assets "
            "in Iraq and Syria in Q4 2025. US-Iran tensions are elevated but have not "
            "escalated to direct military exchange.\n"
            "- Lloyd's of London hull-war insurance premiums for tankers transiting the "
            "Gulf of Oman have risen approximately 15% since September 2025.\n"
            "- The WTI NYMEX forward curve is in mild backwardation: front month $58, "
            "6-month forward approximately $56.\n"
            "- EIA weekly report (Dec 31 2025): US crude inventories 8% below the 5-year "
            "seasonal average."
        ),
    },
    "2026-02-02": {
        "label": "Feb 2, 2026",
        "threshold_usd": 72.0,
        "context_text": (
            "As of February 2 2026:\n"
            "- WTI gained approximately 7% in January, closing near $62, driven by "
            "escalating Persian Gulf tensions.\n"
            "- A US Navy escort mission in the Gulf of Oman was intercepted by Iranian "
            "fast-attack boats on January 28. No shots fired, but the incident was "
            "widely reported and prompted a diplomatic protest from Washington.\n"
            "- OPEC+ called an emergency ministerial consultation for February 10 amid "
            "concerns about supply-chain disruption risk; no production change announced yet.\n"
            "- Goldman Sachs revised its 2026 WTI price target upward to $70–85 in a "
            "February 1 research note, citing a 'geopolitical risk premium re-rating'.\n"
            "- Vessel-tracking data shows tanker transits through the Strait of Hormuz "
            "down approximately 15% week-over-week, as operators seek alternative routings.\n"
            "- Brent/WTI spread widened to $4.50, the largest since early 2024, as "
            "European buyers began bidding up non-Gulf grades.\n"
            "- US intelligence officials stated publicly that Iranian military assets "
            "have been repositioned closer to the Strait of Hormuz."
        ),
    },
    "2026-03-02": {
        "label": "Mar 2, 2026",
        "threshold_usd": 85.0,
        "context_text": (
            "As of March 2 2026:\n"
            "- The US conducted direct airstrikes on Iranian oil-infrastructure targets "
            "on March 1 2026 in response to an Iranian attack on a US carrier group "
            "in the Gulf of Oman on February 26.\n"
            "- Iran declared a partial blockade of the Strait of Hormuz effective "
            "March 1; approximately 20% of global seaborne oil supply transits the Strait.\n"
            "- WTI surged from $62 on February 2 to $71 by March 2 — a 14% move in "
            "one month — and front-month futures gapped up a further $4 at Monday open.\n"
            "- The IEA called an emergency ministerial meeting for March 5 to consider "
            "releasing strategic petroleum reserves.\n"
            "- Saudi Aramco issued force majeure declarations on several customer contracts; "
            "Saudi Arabia activated its emergency supply protocols.\n"
            "- Goldman Sachs issued an updated note on March 1 with a new 2026 WTI target "
            "of $95–115 and flagging a tail-risk scenario of $130 if the blockade persists "
            "beyond 60 days.\n"
            "- WTI NYMEX forward curve has swung into sharp backwardation: front month "
            "$71, 6-month forward $62, signalling market expectation of eventual resolution."
        ),
    },
}

print("Context snippets defined for:", list(ORIGIN_CONTEXTS.keys()))

Context snippets defined for: ['2026-01-05', '2026-02-02', '2026-03-02']


In [20]:
# ── Run LLMP forecasts (or load from cache) ───────────────────────────────────
#
# We use the LLMP module's internals directly so we can supply a pre-compressed
# history DataFrame rather than going through a DataService.  This is appropriate
# for a playground notebook; a production version would use a registered adapter.
#
# Each origin × 2 variants (bare / with-context) = 6 LLM calls.
# Results are cached to data/energy_llmp_context_forecasts.parquet.

from aieng.forecasting.methods.llm_processes.continuous import (
    ContinuousLLMPredictorConfig,
    _build_system_prompt,
    _build_user_prompt,
    _quantiles_per_step,
    _sample_trajectories,
    _stack_trajectories,
)
from aieng.forecasting.methods.llm_processes.base import serialize_history
from aieng.forecasting.evaluation.prediction import STANDARD_QUANTILES
from aieng.forecasting.evaluation.task import ForecastingTask
from aieng.forecasting.data.models import SeriesMetadata


MODEL       = "gemini/gemini-3-flash-preview"
N_SAMPLES   = 20
HORIZON_B   = 21   # ~21 business days ≈ 30 calendar days
PRECISION   = 2

_WTI_TASK = ForecastingTask(
    task_id="wti_crude_30d",
    target_series_id="wti_crude",
    horizons=list(range(1, HORIZON_B + 1)),
    frequency="B",
    description=(
        "WTI crude oil front-month futures price (USD/bbl), "
        "30 trading-day ahead probabilistic forecast. "
        "Forecast the daily closing price for each of the next "
        f"{HORIZON_B} business days."
    ),
)

_WTI_META = SeriesMetadata(
    series_id="wti_crude",
    description="WTI crude oil front-month futures (CL=F, Adj Close)",
    source="Yahoo Finance",
    units="USD/bbl",
    frequency="B",
)


def _run_one_forecast(
    price_df: pd.DataFrame,
    origin: pd.Timestamp,
    context_text: str | None,
    context_tag: str,
) -> dict:
    """Run LLMP at a single origin and return a dict of arrays."""
    history_df = compress_history(price_df, origin)
    history_str = serialize_history(history_df, precision=PRECISION)

    forecast_start = origin + pd.offsets.BDay(1)
    forecast_end   = origin + pd.offsets.BDay(HORIZON_B)

    system_prompt = _build_system_prompt()
    user_prompt   = _build_user_prompt(
        _WTI_TASK, history_str, _WTI_META,
        forecast_start, forecast_end, HORIZON_B,
        context_text=context_text,
    )

    cfg = ContinuousLLMPredictorConfig(
        model=MODEL,
        n_samples=N_SAMPLES,
        temperature=1.0,
        reasoning_effort="disable",
        context_text=context_text,
        context_tag=context_tag,
    )

    parsed, cost_usd, in_tok, out_tok, failures = _sample_trajectories(
        cfg=cfg, system_prompt=system_prompt, user_prompt=user_prompt
    )
    samples = _stack_trajectories([t.values for t in parsed], n_steps=HORIZON_B)
    q_grid  = _quantiles_per_step(samples)   # (HORIZON_B, len(STANDARD_QUANTILES))

    dates = pd.bdate_range(start=origin + pd.offsets.BDay(1), periods=HORIZON_B)

    rows = []
    for h_idx in range(HORIZON_B):
        row: dict = {
            "origin":      origin,
            "context_tag": context_tag,
            "forecast_date": dates[h_idx],
            "horizon":     h_idx + 1,
            "median":      float(q_grid[h_idx, STANDARD_QUANTILES.index(0.50)]),
            "cost_usd":    cost_usd,
        }
        for qi, q in enumerate(STANDARD_QUANTILES):
            row[f"q{int(q * 100):02d}"] = float(q_grid[h_idx, qi])
        rows.append(row)

    print(
        f"    origin={origin.date()} tag={context_tag:12s} "
        f"cost=${cost_usd:.4f}  failures={failures}/{N_SAMPLES}"
    )
    return {"rows": rows, "samples": samples.tolist()}


def run_all_forecasts(price_df: pd.DataFrame, cache_path: Path) -> pd.DataFrame:
    """Run (or load) all 6 LLMP forecasts; return a single flat DataFrame."""
    if cache_path.exists():
        df = pd.read_parquet(cache_path)
        df["origin"]        = pd.to_datetime(df["origin"])
        df["forecast_date"] = pd.to_datetime(df["forecast_date"])
        print(f"Loaded {len(df)} LLMP forecast rows from cache.")
        return df

    all_rows: list[dict] = []
    print("Running LLMP forecasts (6 API calls)...")
    for origin in ORIGINS:
        key = origin.strftime("%Y-%m-%d")
        ctx = ORIGIN_CONTEXTS[key]
        for tag, text in [("bare", None), ("context", ctx["context_text"])]:
            result = _run_one_forecast(price_df, origin, text, tag)
            all_rows.extend(result["rows"])

    df = pd.DataFrame(all_rows)
    df.to_parquet(cache_path, index=False)
    print(f"Saved {len(df)} rows to {cache_path}")
    return df


llmp_df = run_all_forecasts(price_df, LLMP_CACHE)
print(f"\nOrigins: {sorted(llmp_df['origin'].dt.date.unique())}")
print(f"Tags:    {sorted(llmp_df['context_tag'].unique())}")
llmp_df.head(6)

Loaded 126 LLMP forecast rows from cache.

Origins: [datetime.date(2026, 1, 5), datetime.date(2026, 2, 2), datetime.date(2026, 3, 2)]
Tags:    ['bare', 'context']


,origin,context_tag,forecast_date,horizon,median,cost_usd,q05,q10,q20,q30,q40,q50,q60,q70,q80,q90,q95
0,2026-01-05,bare,2026-01-06,1,58.450,0.07594,58.1500,58.420,58.450,58.450,58.450,58.450,58.450,58.450,58.624,58.650,58.6595
1,2026-01-05,bare,2026-01-07,2,58.820,0.07594,58.3955,58.410,58.620,58.720,58.780,58.820,58.820,58.820,59.120,59.123,59.1530
2,2026-01-05,bare,2026-01-08,3,59.080,0.07594,58.1935,58.300,58.590,58.769,58.846,59.080,59.150,59.150,59.150,59.180,59.4515
3,2026-01-05,bare,2026-01-09,4,58.835,0.07594,58.0910,58.120,58.394,58.700,58.724,58.835,58.874,59.158,59.226,59.384,59.4315
4,2026-01-05,bare,2026-01-12,5,59.165,0.07594,57.8850,57.937,58.426,58.550,59.052,59.165,59.232,59.323,59.420,59.787,59.8515
5,2026-01-05,bare,2026-01-13,6,59.280,0.07594,57.9150,58.400,58.686,58.959,59.092,59.280,59.368,59.680,59.904,60.120,60.1200


---

## Act 5 — Trajectory: Can the Model See the Move Coming?

Each panel below shows the 30-day forecast fan from one origin date.
Three forecast traces:
- **Grey** — Prophet (statistical baseline; history only)
- **Orange** — LLMP, history only (same information as Prophet, different model family)
- **Green** — LLMP with context (public geopolitical information available on that date)

The **solid blue line** is the realized WTI price (actual outcome).
Shading shows 50% and 90% credible intervals for the LLMP forecasts.

In [21]:
def _add_fan(
    fig: go.Figure,
    dates: "pd.Series",
    lower: "pd.Series",
    upper: "pd.Series",
    median: "pd.Series",
    color: str,
    opacity_band: float,
    name: str,
    show_legend: bool,
    legendgroup: str,
    row: int,
    col: int,
) -> None:
    """Add a forecast fan (CI band + median line) to a subplot panel."""
    # CI shading (fill between lower and upper)
    fig.add_trace(
        go.Scatter(
            x=pd.concat([dates, dates[::-1]]),
            y=pd.concat([lower, upper[::-1]]),
            fill="toself",
            fillcolor=color,
            opacity=opacity_band,
            line=dict(width=0),
            mode="lines",
            showlegend=False,
        ),
        row=row, col=col,
    )
    # Median line
    fig.add_trace(
        go.Scatter(
            x=dates, y=median,
            line=dict(color=color, width=2),
            name=name if show_legend else None,
            showlegend=show_legend,
            legendgroup=legendgroup,
        ),
        row=row, col=col,
    )


def make_trajectory_figure(
    price_df: pd.DataFrame,
    prophet_traj_df: pd.DataFrame,
    llmp_df: pd.DataFrame,
    origins: list[pd.Timestamp],
) -> go.Figure:
    """Three-column subplot: one panel per origin, like-for-like trajectory fan comparison.

    All three methods (Prophet, LLMP-bare, LLMP-with-context) are shown as a
    CI-band + median line over the full 21-business-day forecast horizon, making
    the comparison visually consistent.
    """
    labels = [ORIGIN_CONTEXTS[o.strftime("%Y-%m-%d")]["label"] for o in origins]

    fig = psp.make_subplots(
        rows=1, cols=3,
        subplot_titles=labels,
        shared_yaxes=False,
        horizontal_spacing=0.06,
    )

    for col, origin in enumerate(origins, start=1):
        # ── History (60 days pre-origin) ──────────────────────────────────────
        hist_start = origin - pd.Timedelta(days=60)
        hist = price_df[price_df.index >= hist_start].loc[:origin]
        fig.add_trace(
            go.Scatter(
                x=hist.index, y=hist["price"],
                line=dict(color=CLR_HISTORY, width=2),
                name="History" if col == 1 else None,
                showlegend=(col == 1),
                legendgroup="history",
            ),
            row=1, col=col,
        )

        # ── Actuals post-origin ────────────────────────────────────────────────
        res_date, _ = resolution_price(price_df, origin)
        actuals = price_df[(price_df.index > origin) & (price_df.index <= res_date)]
        fig.add_trace(
            go.Scatter(
                x=actuals.index, y=actuals["price"],
                line=dict(color=CLR_ACTUAL, width=2.5),
                name="Actual" if col == 1 else None,
                showlegend=(col == 1),
                legendgroup="actual",
            ),
            row=1, col=col,
        )

        # ── Prophet trajectory fan ─────────────────────────────────────────────
        pt = prophet_traj_df[prophet_traj_df["origin"] == origin].sort_values("forecast_date")
        if not pt.empty:
            _add_fan(
                fig,
                dates=pt["forecast_date"],
                lower=pt["yhat_lower"],
                upper=pt["yhat_upper"],
                median=pt["yhat"],
                color=CLR_PROPHET,
                opacity_band=0.15,
                name="Prophet 95% CI",
                show_legend=(col == 1),
                legendgroup="prophet",
                row=1, col=col,
            )

        # ── LLMP fans (bare + context) ─────────────────────────────────────────
        for tag, clr, name in [
            ("bare",    CLR_LLMP_BARE, "LLMP — history only"),
            ("context", CLR_LLMP_CTX,  "LLMP — with context"),
        ]:
            sub = llmp_df[(llmp_df["origin"] == origin) & (llmp_df["context_tag"] == tag)].sort_values("forecast_date")
            if sub.empty:
                continue

            # 90% CI (outer, light)
            _add_fan(
                fig,
                dates=sub["forecast_date"],
                lower=sub["q05"],
                upper=sub["q95"],
                median=sub["median"] if "median" in sub.columns else sub["q50"],
                color=clr,
                opacity_band=0.10,
                name=name,
                show_legend=(col == 1),
                legendgroup=f"llmp_{tag}",
                row=1, col=col,
            )
            # 50% CI (inner, darker)
            if "q25" in sub.columns and "q75" in sub.columns:
                fig.add_trace(
                    go.Scatter(
                        x=pd.concat([sub["forecast_date"], sub["forecast_date"][::-1]]),
                        y=pd.concat([sub["q25"], sub["q75"][::-1]]),
                        fill="toself",
                        fillcolor=clr,
                        opacity=0.20,
                        line=dict(width=0),
                        mode="lines",
                        showlegend=False,
                    ),
                    row=1, col=col,
                )

        # ── Origin marker ──────────────────────────────────────────────────────
        fig.add_vline(
            x=origin.timestamp() * 1000,
            line=dict(color="#888", dash="dash", width=1),
            row=1, col=col,
        )

    fig.update_layout(
        title=dict(
            text="30-Day WTI Forecast Trajectories — Prophet vs. LLMP vs. LLMP + Context",
            font=dict(size=15),
        ),
        height=420,
        width=1200,
        legend=dict(orientation="h", y=-0.18),
        template="plotly_white",
        margin=dict(t=60, b=90),
    )
    fig.update_yaxes(title_text="WTI (USD/bbl)", col=1)
    return fig


make_trajectory_figure(price_df, prophet_traj_df, llmp_df, ORIGINS).show()

### What to look for

- **January origin**: Prophet barely covers the outcome (within CI). Does LLMP-with-context
  shift the median upward given the rising-tension signals? A modest upward shift is the
  "right" answer — the context signals elevated risk without a clear catalyst yet.
- **February origin**: Prophet misses (actual $75, CI upper ~$69). Does LLMP-with-context
  produce a median meaningfully higher than LLMP-bare? The context describes a naval incident
  and analyst upgrades — strong reasons to expect further upside.
- **March origin**: The most dramatic case. Conflict is active. Does LLMP-with-context
  shift its distribution toward $100? Anything in the $80–95 range would represent a
  major improvement over Prophet's $61.

---

## Act 6 — Binary: Will Oil Prices Stay Stable This Week?

A single, intuitive question: **"Will WTI price remain within ±$3/bbl of today's level over the next 5 trading days?"**

This is the kind of question a risk manager or trader would ask every Monday morning.
"Normal" weekly oil price movement is ±1–3%, so ±$3 on a ~$60–70 base represents roughly a calm week.

Each method turns its continuous forecast into a probability for this event.
We score each method with the **Brier Score**:

$$\text{BS} = (P(\text{stable}) - \text{outcome})^2$$

where outcome = 1 if prices actually stayed stable, 0 if they broke out.
**Lower is better. BS = 0 is perfect. BS = 0.25 is equivalent to always saying 50/50.**

| Origin | Actual 5-day max move | Outcome |
|---|---|---|
| Jan 5, 2026 | +$1.18 (stayed calm) | **Stable ✓** |
| Feb 2, 2026 | +$2.22 (some tension, held) | **Stable ✓** |
| Mar 2, 2026 | +$23.54 (conflict, explosive) | **Unstable ✗** |

In [22]:
import scipy.interpolate
import scipy.stats

# ── Parameters for the binary question ───────────────────────────────────────
STABILITY_THRESHOLD = 3.0   # ±$3/bbl
STABILITY_HORIZON   = 5     # business days (≈ 1 trading week)

# ── Ground-truth: did prices actually stay stable? ────────────────────────────
def check_stable_outcome(
    price_df: pd.DataFrame,
    origin: pd.Timestamp,
    threshold: float,
    horizon_bdays: int,
) -> tuple[int, float]:
    """Return (outcome, max_abs_move) over horizon_bdays after origin.

    outcome = 1 if ALL daily closes stayed within ±threshold of the origin price.
    outcome = 0 if any day broke out of that range.
    """
    origin_price = float(price_df[price_df.index >= origin].iloc[0]["price"])
    future_days  = price_df[price_df.index > origin].iloc[:horizon_bdays]
    moves        = (future_days["price"] - origin_price).abs()
    max_move     = float(moves.max())
    return (1 if max_move <= threshold else 0), max_move


# ── P(stable) from LLMP quantile distribution at horizon h ───────────────────
def llmp_prob_stable(
    llmp_sub: pd.DataFrame,
    origin_price: float,
    threshold: float,
    horizon: int = STABILITY_HORIZON,
) -> float:
    """P(|price_h - origin_price| ≤ threshold) from the LLMP quantile grid.

    Uses linear interpolation of the quantile CDF to estimate the probability
    mass inside [origin - threshold, origin + threshold].
    """
    row = llmp_sub[llmp_sub["horizon"] == horizon]
    if row.empty:
        return float("nan")
    row = row.iloc[0]

    q_levels = STANDARD_QUANTILES
    q_vals   = [float(row[f"q{int(q * 100):02d}"]) for q in q_levels]

    # Build a monotone CDF by sorting (defensive against non-monotone samples)
    pairs = sorted(zip(q_vals, q_levels))
    vals, cumprobs = zip(*pairs)

    cdf = scipy.interpolate.interp1d(
        vals, cumprobs, kind="linear", bounds_error=False, fill_value=(0.0, 1.0)
    )
    p_below_upper = float(cdf(origin_price + threshold))
    p_below_lower = float(cdf(origin_price - threshold))
    return float(np.clip(p_below_upper - p_below_lower, 0.0, 1.0))


# ── P(stable) from Prophet trajectory at horizon h ───────────────────────────
def prophet_prob_stable(
    prophet_traj_sub: pd.DataFrame,
    origin_price: float,
    threshold: float,
    horizon: int = STABILITY_HORIZON,
) -> float:
    """P(|price_h - origin_price| ≤ threshold) from Prophet's Gaussian-like CI.

    Prophet's 95% CI implies σ ≈ (upper − lower) / (2 × 1.96).
    """
    row = prophet_traj_sub[prophet_traj_sub["horizon"] == horizon]
    if row.empty:
        return float("nan")
    row = row.iloc[0]

    sigma = (float(row["yhat_upper"]) - float(row["yhat_lower"])) / (2 * 1.96)
    if sigma <= 0:
        return 1.0 if abs(float(row["yhat"]) - origin_price) <= threshold else 0.0

    p_below_upper = scipy.stats.norm.cdf(origin_price + threshold, loc=row["yhat"], scale=sigma)
    p_below_lower = scipy.stats.norm.cdf(origin_price - threshold, loc=row["yhat"], scale=sigma)
    return float(np.clip(p_below_upper - p_below_lower, 0.0, 1.0))


# ── Assemble binary results table ─────────────────────────────────────────────
binary_rows = []
for origin in ORIGINS:
    key          = origin.strftime("%Y-%m-%d")
    origin_price = float(price_df[price_df.index >= origin].iloc[0]["price"])
    outcome, max_move = check_stable_outcome(price_df, origin, STABILITY_THRESHOLD, STABILITY_HORIZON)

    pt_sub = prophet_traj_df[prophet_traj_df["origin"] == origin]
    p_prob = prophet_prob_stable(pt_sub, origin_price, STABILITY_THRESHOLD)
    binary_rows.append({
        "origin": key, "label": ORIGIN_CONTEXTS[key]["label"],
        "origin_price": origin_price, "max_move": max_move,
        "outcome": outcome, "method": "Prophet",
        "prob": p_prob, "brier": (p_prob - outcome) ** 2,
    })

    for tag in ["bare", "context"]:
        sub    = llmp_df[(llmp_df["origin"] == origin) & (llmp_df["context_tag"] == tag)]
        l_prob = llmp_prob_stable(sub, origin_price, STABILITY_THRESHOLD)
        binary_rows.append({
            "origin": key, "label": ORIGIN_CONTEXTS[key]["label"],
            "origin_price": origin_price, "max_move": max_move,
            "outcome": outcome, "method": f"LLMP — {tag}",
            "prob": l_prob, "brier": (l_prob - outcome) ** 2,
        })

binary_df = pd.DataFrame(binary_rows)

print(f"Stability question: |Δprice over {STABILITY_HORIZON} days| ≤ ${STABILITY_THRESHOLD}")
print()
display_cols = ["label", "outcome", "max_move", "method", "prob", "brier"]
print(binary_df[display_cols].to_string(index=False, float_format="{:.3f}".format))
print()
print("Brier score interpretation: 0 = perfect, 0.25 = always-50% baseline, 1 = perfectly wrong")

      label  threshold  actual_price         method         prob    brier
Jan 5, 2026       65.0     65.139999    LLMP — bare 0.000000e+00 1.000000
Jan 5, 2026       65.0     65.139999 LLMP — context 0.000000e+00 1.000000
Jan 5, 2026       65.0     65.139999        Prophet 3.344606e-02 0.934227
Feb 2, 2026       72.0     74.660004    LLMP — bare 0.000000e+00 1.000000
Feb 2, 2026       72.0     74.660004 LLMP — context 1.000000e+00 0.000000
Feb 2, 2026       72.0     74.660004        Prophet 4.380632e-03 0.991258
Mar 2, 2026       85.0    100.120003    LLMP — bare 0.000000e+00 1.000000
Mar 2, 2026       85.0    100.120003 LLMP — context 8.574365e-01 0.020324
Mar 2, 2026       85.0    100.120003        Prophet 2.894194e-09 1.000000


In [11]:
METHOD_COLORS = {
    "Prophet":        CLR_PROPHET,
    "LLMP — bare":    CLR_LLMP_BARE,
    "LLMP — context": CLR_LLMP_CTX,
}

# ── Figure: 2 rows × 3 cols ───────────────────────────────────────────────────
# Row 1: Actual 5-day price path vs. stability band (visual evidence)
# Row 2: P(stable) probability bars with Brier score (the score card)

LABELS = [ORIGIN_CONTEXTS[o.strftime("%Y-%m-%d")]["label"] for o in ORIGINS]

fig = psp.make_subplots(
    rows=2, cols=3,
    subplot_titles=LABELS + ["", "", ""],
    row_heights=[0.60, 0.40],
    vertical_spacing=0.14,
    horizontal_spacing=0.08,
)

for col, origin in enumerate(ORIGINS, start=1):
    key          = origin.strftime("%Y-%m-%d")
    origin_price = float(price_df[price_df.index >= origin].iloc[0]["price"])
    sub_bin      = binary_df[binary_df["origin"] == key]
    outcome      = int(sub_bin.iloc[0]["outcome"])

    # ── Row 1: Price path ─────────────────────────────────────────────────────
    # Brief history (14 days pre-origin, grey)
    hist = price_df[price_df.index >= origin - pd.Timedelta(days=20)].loc[:origin]
    fig.add_trace(
        go.Scatter(
            x=hist.index, y=hist["price"],
            line=dict(color=CLR_HISTORY, width=1.5),
            name="History" if col == 1 else None,
            showlegend=(col == 1), legendgroup="hist_b",
        ),
        row=1, col=col,
    )

    # 5-day actual price path (green if stable, red if unstable)
    future5 = price_df[price_df.index > origin].iloc[:STABILITY_HORIZON]
    path_clr = CLR_ACTUAL if outcome == 1 else CLR_CONFLICT
    path_label = ("Actual — STABLE ✓" if outcome == 1 else "Actual — UNSTABLE ✗")
    fig.add_trace(
        go.Scatter(
            x=future5.index, y=future5["price"],
            line=dict(color=path_clr, width=2.5),
            mode="lines+markers", marker=dict(size=7),
            name=path_label if col == 1 else None,
            showlegend=(col == 1), legendgroup=f"path_{outcome}",
        ),
        row=1, col=col,
    )

    # Stability band (shaded rectangle)
    x_band = [origin, future5.index[-1], future5.index[-1], origin]
    y_band = [
        origin_price - STABILITY_THRESHOLD, origin_price - STABILITY_THRESHOLD,
        origin_price + STABILITY_THRESHOLD, origin_price + STABILITY_THRESHOLD,
    ]
    fig.add_trace(
        go.Scatter(
            x=x_band, y=y_band,
            fill="toself", fillcolor="rgba(100,200,100,0.10)",
            line=dict(color="rgba(100,200,100,0.4)", width=1, dash="dot"),
            mode="lines",
            name=f"Stability zone ±${STABILITY_THRESHOLD:.0f}" if col == 1 else None,
            showlegend=(col == 1), legendgroup="band",
        ),
        row=1, col=col,
    )

    # Origin marker
    fig.add_vline(
        x=origin.timestamp() * 1000,
        line=dict(color="#aaa", dash="dash", width=1),
        row=1, col=col,
    )

    fig.update_yaxes(title_text="WTI (USD/bbl)" if col == 1 else None, row=1, col=col)

    # ── Row 2: P(stable) bars + Brier scores ─────────────────────────────────
    methods_ordered = ["Prophet", "LLMP — bare", "LLMP — context"]
    for method in methods_ordered:
        mrow = sub_bin[sub_bin["method"] == method]
        if mrow.empty:
            continue
        prob  = float(mrow.iloc[0]["prob"])
        brier = float(mrow.iloc[0]["brier"])
        clr   = METHOD_COLORS[method]

        fig.add_trace(
            go.Bar(
                x=[prob],
                y=[method],
                orientation="h",
                marker_color=clr,
                marker_opacity=0.80,
                text=[f"  {prob:.0%}  (BS={brier:.2f})"],
                textposition="outside",
                textfont=dict(size=10),
                name=method if col == 1 else None,
                showlegend=(col == 1),
                legendgroup=f"method_{method}",
            ),
            row=2, col=col,
        )

    # Outcome line
    outcome_label = "Stable ✓" if outcome == 1 else "Unstable ✗"
    outcome_clr   = "#2ca02c" if outcome == 1 else "#d62728"
    fig.add_vline(
        x=float(outcome),
        line=dict(color=outcome_clr, dash="dot", width=2),
        annotation_text=outcome_label,
        annotation_position="top right" if outcome == 1 else "top left",
        annotation_font=dict(size=10, color=outcome_clr),
        row=2, col=col,
    )

    fig.update_xaxes(
        range=[0, 1.25], tickformat=".0%",
        title_text="P(stable)" if col == 1 else None,
        row=2, col=col,
    )

fig.update_layout(
    title=dict(
        text=(
            f"Will oil price stay within ±${STABILITY_THRESHOLD:.0f}/bbl over the next "
            f"{STABILITY_HORIZON} trading days? — Probability and Brier Score by Method"
        ),
        font=dict(size=14),
    ),
    height=560,
    width=1150,
    template="plotly_white",
    legend=dict(orientation="h", y=-0.12, font=dict(size=10)),
    barmode="relative",
    margin=dict(t=70, b=80),
)
fig.show()

---

## Act 7 — Causal: What Did the Context Actually Do?

The trajectory and binary charts tell us *whether* the context helped.
This section asks *by how much* — and sets up the bigger question:
what would a proper forecasting agent do with more than a text snippet?

In [12]:
# ── Median shift: bare → context at the 30-day horizon ───────────────────────
shift_rows = []
for origin in ORIGINS:
    key  = origin.strftime("%Y-%m-%d")
    sub  = llmp_df[(llmp_df["origin"] == origin) & (llmp_df["horizon"] == HORIZON_B)]
    bare = sub[sub["context_tag"] == "bare"]
    ctx  = sub[sub["context_tag"] == "context"]
    bare_med = float(bare["median"].iloc[0]) if not bare.empty else float("nan")
    ctx_med  = float(ctx["median"].iloc[0])  if not ctx.empty  else float("nan")
    _, actual = resolution_price(price_df, origin)
    origin_price = float(price_df[price_df.index >= origin].iloc[0]["price"])

    shift_rows.append({
        "Origin":             ORIGIN_CONTEXTS[key]["label"],
        "Actual price":       f"${actual:.0f}",
        "LLMP bare median":   f"${bare_med:.1f}",
        "LLMP+ctx median":    f"${ctx_med:.1f}",
        "Context shift":      f"+${ctx_med - bare_med:.1f}" if ctx_med >= bare_med else f"${ctx_med - bare_med:.1f}",
        "Actual vs. bare":    f"{actual - bare_med:+.1f}",
        "Actual vs. +ctx":    f"{actual - ctx_med:+.1f}",
        "Context in context": ORIGIN_CONTEXTS[key]["context_text"].split("\n")[0],
    })

shift_df = pd.DataFrame(shift_rows)
display_cols = ["Origin", "LLMP bare median", "LLMP+ctx median", "Context shift", "Actual price", "Actual vs. bare", "Actual vs. +ctx"]
shift_df[display_cols]

ValueError: 
    Invalid value of type 'builtins.str' received for the 'yref' property of layout.annotation
        Received value: 'y1 domain'

    The 'yref' property is an enumeration that may be specified as:
      - One of the following enumeration values:
            ['paper']
      - A string that matches one of the following regular expressions:
            ['^y([2-9]|[1-9][0-9]+)?( domain)?$']

### Reading the table

- **Context shift** is how much the 30-day median moves when context is added.
  A positive shift means the model correctly anchored on upward price pressure.
- **Actual vs. bare / Actual vs. +ctx** is the residual error. Smaller (closer to 0) is better.
- The March origin is the most telling: even a large positive context shift still
  undershoots the actual $100 price, because no purely-text-based model can fully
  price in a live supply disruption without numerical data to anchor on.

### This is the case for a Track 2 Analyst Agent

The LLMP with a text snippet is the *minimum viable context-aware forecaster*:
it can move in the right direction, but it works from headlines, not data.

| What we did here | What a Track 2 Analyst Agent would do |
|---|---|
| Manually curated a text snippet | Retrieves context automatically — news, futures, sentiment |
| Context is unverified prose | Queries DataService; validates signals in code |
| Single point-of-view forecast | Runs explicit scenarios ("blockade persists 60 days" vs. "resolves in 2 weeks") |
| No explanation | Produces a written narrative: sources, assumptions, confidence level |
| No follow-up | Conversational — answers "what would change your forecast?" |

> *"The LLMP with context is already seeing something Prophet cannot.
> The Analyst Agent is the version that knows what to look for and explains why."*

---

## Evaluation Summary

Point MAE at the 30-day resolution horizon, and directional accuracy
(did the forecast median correctly call whether price was higher or lower
than the origin price?).

In [13]:
eval_rows = []

for origin in ORIGINS:
    key = origin.strftime("%Y-%m-%d")
    _, actual = resolution_price(price_df, origin)
    origin_price = float(price_df[price_df.index >= origin].iloc[0]["price"])
    true_direction = actual > origin_price

    # Prophet
    p_row = prophet_row_at_origin(prophet_df, origin)
    p_mae = abs(p_row["yhat"] - actual)
    eval_rows.append({
        "Origin":     ORIGIN_CONTEXTS[key]["label"],
        "Method":     "Prophet",
        "Forecast":   f"${p_row['yhat']:.1f}",
        "Actual":     f"${actual:.1f}",
        "MAE ($)": f"{p_mae:.1f}",
        "Dir. correct": "✓" if (p_row["yhat"] > origin_price) == true_direction else "✗",
        "Inside CI": str(p_row["inside_ci"]),
    })

    # LLMP
    for tag in ["bare", "context"]:
        sub = llmp_df[(llmp_df["origin"] == origin) & (llmp_df["context_tag"] == tag) & (llmp_df["horizon"] == HORIZON_B)]
        if sub.empty:
            continue
        med = float(sub.iloc[0]["median"])
        mae = abs(med - actual)
        eval_rows.append({
            "Origin":     ORIGIN_CONTEXTS[key]["label"],
            "Method":     f"LLMP — {tag}",
            "Forecast":   f"${med:.1f}",
            "Actual":     f"${actual:.1f}",
            "MAE ($)": f"{mae:.1f}",
            "Dir. correct": "✓" if (med > origin_price) == true_direction else "✗",
            "Inside CI": "—",
        })

eval_df = pd.DataFrame(eval_rows)
eval_df

,Origin,Method,Forecast,Actual,MAE ($),Dir. correct,Inside CI
0,"Jan 5, 2026",Prophet,$57.5,$65.1,7.6,✗,True
1,"Jan 5, 2026",LLMP — bare,$61.0,$65.1,4.2,✓,—
2,"Jan 5, 2026",LLMP — context,$58.0,$65.1,7.1,✗,—
3,"Feb 2, 2026",Prophet,$60.9,$74.7,13.7,✗,False
4,"Feb 2, 2026",LLMP — bare,$61.8,$74.7,12.9,✗,—
5,"Feb 2, 2026",LLMP — context,$78.5,$74.7,3.9,✓,—
6,"Mar 2, 2026",Prophet,$61.3,$100.1,38.8,✗,False
7,"Mar 2, 2026",LLMP — bare,$68.4,$100.1,31.7,✗,—
8,"Mar 2, 2026",LLMP — context,$96.3,$100.1,3.8,✓,—


### Binary evaluation — Brier score summary

**Question:** Will price stay within ±$3/bbl over the next 5 trading days?
**Brier Score** = (P(stable) − outcome)².  Lower is better.  0.25 = always predicting 50%.

In [14]:
pivot = (
    binary_df
    .assign(
        prob_str=lambda d: d["prob"].map(lambda p: f"{p:.0%}"),
        brier_str=lambda d: d["brier"].map(lambda b: f"{b:.3f}"),
        summary=lambda d: d["prob_str"] + "  (BS " + d["brier_str"] + ")",
    )
    .pivot_table(index="label", columns="method", values="summary", aggfunc="first")
)
pivot.index.name = "Origin"
pivot.columns.name = None

# Prepend outcome + max_move columns
outcome_col = binary_df.groupby("label").first()[["outcome", "max_move"]].reindex(pivot.index)
outcome_col.columns = ["Stable? (1=yes)", "Max |Δ| ($)"]
outcome_col["Max |Δ| ($)"] = outcome_col["Max |Δ| ($)"].map("{:.2f}".format)

result = pd.concat([outcome_col, pivot], axis=1)
result

,Threshold exceeded?,LLMP — bare — brier_str,LLMP — context — brier_str,Prophet — brier_str,LLMP — bare — prob_pct,LLMP — context — prob_pct,Prophet — prob_pct
Origin,,,,,,,
"Feb 2, 2026",YES,1.0,0.0,0.991,0.0,100.0,0.4
"Jan 5, 2026",YES,1.0,1.0,0.934,0.0,0.0,3.3
"Mar 2, 2026",YES,1.0,0.02,1.0,0.0,85.7,0.0
